# Similarity Model

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

In [ ]:
pd.set_option('display.max_colwidth', None)

# Building Similarity Model
def train_similarity_model(data_filepath, composite_cols=None, n_components=40):
    """
    Loads preprocessed data, creates a composite text feature, and trains a TF-IDF
    + TruncatedSVD model. Returns the DataFrame, vectorizer, SVD transformer,
    and the cosine similarity matrix.
    """
    # Load data
    df = pd.read_csv(data_filepath, dtype=str)
    df = df.fillna("")
    
    # Columns to create combined_text
    existing_cols = ["DESCRIPTION", "Attribut1", "Characteristic", "Rating"]
    df['combined_text'] = df[existing_cols].apply(
        lambda row: ", ".join([str(x).strip() for x in row if str(x).strip() != ""]),
        axis=1
    )
    
    # combined_text for vectorization
    text_data = df['combined_text']
    
    # TF-IDF Vectorization 
    vectorizer = TfidfVectorizer(
        stop_words='english', 
        ngram_range=(1,2), 
        max_df=0.85, 
        min_df=2, 
        sublinear_tf=True
    )
    tfidf_matrix = vectorizer.fit_transform(text_data)
    
    # Dimensionality reduction using TruncatedSVD (LSA)
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    tfidf_reduced = svd.fit_transform(tfidf_matrix)
    
    # Cosine similarity on reduced vectors
    cos_sim_matrix = cosine_similarity(tfidf_reduced, tfidf_reduced)
    
    return df, vectorizer, svd, cos_sim_matrix

def get_top_n_similar(part_index, cos_sim_matrix, n=5):
    """
    Given a part index, returns a list of (index, similarity score) tuples for the top n similar parts.
    Excludes the part itself.
    """
    sim_scores = list(enumerate(cos_sim_matrix[part_index]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_n = sim_scores[1:n+1]  # Exclude the part itself
    return top_n


In [6]:
# Training & Testing : with ID's
def recommend_for_part(df, cos_sim_matrix, part_identifier, n=5):
    """
    Given a part ID , returns :
      1. The input part's details (ID and DESCRIPTION).
      2. Detailed recommendations including the alternative part's ID, Similarity Score,
         DESCRIPTION, and Combined Text.
    """
    # Index from part_identifier
    if isinstance(part_identifier, int):
        idx = part_identifier
    else:
        idx_list = df.index[df['ID'] == part_identifier].tolist()
        if not idx_list:
            print(f"Part ID '{part_identifier}' not found!")
            return None, None
        idx = idx_list[0]
    
    #  Input part's details (ID and full DESCRIPTION)
    input_part_df = df.loc[[idx], ['ID', 'DESCRIPTION']]
    
    # Get top n similar parts
    top_sim = get_top_n_similar(idx, cos_sim_matrix, n=n)
    
    recommendations = []
    for i, score in top_sim:
        recommendations.append({
            "ID": df.iloc[i]['ID'],
            "Similarity Score": np.round(score, 3),
            "DESCRIPTION": df.iloc[i]['DESCRIPTION']
        })
    recs_df = pd.DataFrame(recommendations)
    return input_part_df, recs_df


# Execution: Training & Testing

if __name__ == "__main__":
    # Train the model (adjust file path as needed)
    data_filepath = "Parts_processed.csv"
    df, vectorizer, svd, cos_sim_matrix = train_similarity_model(data_filepath)
    
    # User based testing: Manually input a Part ID
    user_input = input("Enter a Part ID for recommendations: ").strip().upper()
    
    input_part_df, detailed_recs = recommend_for_part(df, cos_sim_matrix, user_input, n=5)
    
    if input_part_df is not None and detailed_recs is not None:
        print("\nInput Part Information:")
        display(input_part_df)  
        
        print(f"\nDetailed Recommendations for Part ID: {user_input}")
        display(detailed_recs)



Input Part Information:


,ID,DESCRIPTION
0,A1,"Indicator Red Fast Movement 1.6A 250V Holder Plastic 5 X 20mm Ceramic Box CCC/PSE/VDE/cULus Electric Indicator, Very Fast Blow, 1.6A, 250VAC, 1500A (IR), Inline/holder, 5x20mm"



Detailed Recommendations for Part ID: A1


,ID,Similarity Score,DESCRIPTION
0,A253,0.864,"Indicator Red Fast Movement 1.6A 250V Holder Plastic 5 X 20mm Ceramic Bulk CCC/CE/CSA/KC/PSE/SEMKO/UL/VDE Electric Indicator, Fast Blow, 1.6A, 250VAC, 1500A (IR), Inline/holder, 5x20mm"
1,A254,0.827,"Indicator Red Fast Movement 1.6A 250V Holder Plastic Melf 5 X 20mm Ceramic CCC/CE/CSA/PSE/SEMKO/UL/VDE Electric Indicator, Fast Blow, 1.6A, 250VAC, 1500A (IR), Inline/holder"
2,A91,0.809,"Red Indicator, 5 X 20 mm, Quick-Movement F, L, 250 VAC Electric Indicator, Fast Blow, 1.6A, 250VAC, 35A (IR), Inline/holder, 5x20mm"
3,A252,0.785,"Indicator Red Fast Movement 1.6A 250V Axial 5 X 20mm Ceramic Bulk CCC/CE/CSA/KC/PSE/SEMKO/UL/VDE Electric Indicator, Fast Blow, 1.6A, 250VAC, 1500A (IR), Through Hole, 5x20mm"
4,A458,0.766,"Indicator Red Fast Movement 1.6A 250V Holder Plastic Melf 6.3 X 32mm Glass CSA/UL Electric Indicator, Fast Blow, 1.6A, 250VAC, 100A (IR), Inline/holder"


#### Part Similarity Recommendation Model
- This model suggests the top 5 alternative part IDs based on a given part input by analyzing combined textual features from each part.
- Feature Statergy :
   - Combines multiple descriptive columns (e.g., `DESCRIPTION`, `Attribut1`, `Characteristic`, `Rating`) into a single `combined_text` field for comprehensive feature representation.
- Feature Extraction:
   - TF-IDF Vectorization Converts text into vector features.
   - N-gram range of (1, 2) to capture both unigrams and bigrams.
   - `max_df=0.85` and `min_df=2` to filter out overly common or rare terms.
- Feature importance: Truncated SVD 
   - Reduces the dimensionality of the TF-IDF matrix to 40 components, retaining the most meaningful semantic information.
- Used similarity scores between parts to recommending similar parts.
- Takes input from user and Retrieves the top 5 similar parts (excluding the input part itself) based on cosine similarity scores.


